In [ ]:
import pandas as pd
import panel as pn
from sqlalchemy import create_engine, text

pn.extension()

USUARIO = 'postgres'
SENHA = '1234' 
HOST = 'localhost'
PORTA = '5432'
BANCO = 'sistema_mentoria_fbd'
string_conexao = f'postgresql+psycopg2://{USUARIO}:{SENHA}@{HOST}:{PORTA}/{BANCO}'
engine = create_engine(string_conexao)

with engine.connect() as conexao:
    query_select = text("SELECT * FROM SESSAO ORDER BY id_sessao DESC")
    resultado = conexao.execute(query_select)
    df_sessoes = pd.DataFrame(resultado.fetchall(), columns=resultado.keys())

input_mentor = pn.widgets.TextInput(name='Matrícula Mentor', width=150)
input_disciplina = pn.widgets.TextInput(name='Código Disciplina', width=150)
input_local = pn.widgets.IntInput(name='ID Local', value=1, width=100)
input_grupo = pn.widgets.IntInput(name='ID Grupo', value=1, width=100)
input_data = pn.widgets.DatePicker(name='Data', width=150)
input_horario = pn.widgets.TextInput(name='Horário (HH:MM)', width=150)

input_id_sessao = pn.widgets.IntInput(name='ID Sessão (Ação)', value=0, width=150)
botao_salvar = pn.widgets.Button(name='Cadastrar Sessão', button_type='primary', width=200)
botao_atualizar = pn.widgets.Button(name='Atualizar Sessão', button_type='warning', width=200)
botao_deletar = pn.widgets.Button(name='Excluir Sessão', button_type='danger', width=200)

tabela_display = pn.pane.DataFrame(df_sessoes, width=1100, max_rows=15)
mensagem = pn.pane.Markdown("")

def atualizar_tabela(conexao):
    res = conexao.execute(query_select)
    tabela_display.object = pd.DataFrame(res.fetchall(), columns=res.keys())

def funcao_inserir(event):
    try:
        with engine.begin() as conexao:
            query = text("""
                INSERT INTO SESSAO (matricula_mentor, codigo_disciplina, id_local, id_grupo, data, horario, presenca_mentor)
                VALUES (:mentor, :disc, :local, :grupo, :data, :horario, TRUE)
            """)
            conexao.execute(query, {
                "mentor": input_mentor.value, "disc": input_disciplina.value,
                "local": input_local.value, "grupo": input_grupo.value,
                "data": input_data.value, "horario": input_horario.value
            })
            atualizar_tabela(conexao)
            mensagem.object = "✅ Sessão cadastrada com sucesso."
    except Exception as e:
        mensagem.object = f"❌ Erro ao cadastrar: {str(e)}"

def funcao_atualizar(event):
    try:
        with engine.begin() as conexao:
            query = text("""
                UPDATE SESSAO 
                SET matricula_mentor = :mentor, codigo_disciplina = :disc, 
                    id_local = :local, id_grupo = :grupo, data = :data, horario = :horario
                WHERE id_sessao = :id_sessao
            """)
            conexao.execute(query, {
                "mentor": input_mentor.value, "disc": input_disciplina.value,
                "local": input_local.value, "grupo": input_grupo.value,
                "data": input_data.value, "horario": input_horario.value,
                "id_sessao": input_id_sessao.value
            })
            atualizar_tabela(conexao)
            mensagem.object = f"⚠️ Sessão {input_id_sessao.value} atualizada."
    except Exception as e:
        mensagem.object = f"❌ Erro ao atualizar: {str(e)}"

def funcao_deletar(event):
    try:
        with engine.begin() as conexao:
            query = text("DELETE FROM SESSAO WHERE id_sessao = :id_sessao")
            conexao.execute(query, {"id_sessao": input_id_sessao.value})
            atualizar_tabela(conexao)
            mensagem.object = f"🗑️ Sessão {input_id_sessao.value} excluída."
    except Exception as e:
        mensagem.object = f"❌ Erro ao excluir: {str(e)}"

botao_salvar.on_click(funcao_inserir)
botao_atualizar.on_click(funcao_atualizar)
botao_deletar.on_click(funcao_deletar)

layout_completo = pn.Column(
    pn.pane.Markdown("## Sistema de Mentoria - CRUD de Sessões"),
    mensagem,
    pn.Row(input_mentor, input_disciplina, input_local, input_grupo, input_data, input_horario),
    pn.Row(botao_salvar),
    pn.layout.Divider(),
    pn.Row(input_id_sessao, botao_atualizar, botao_deletar),
    pn.layout.Divider(),
    tabela_display
)

layout_completo.servable()